# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Detected BASE: C:\Users\Asus\Documents\3rd year\term 1\FDE-Manan-Gupta\nyc-tlc-cbd-pipeline\class_notebooks\flasheats
Database path: C:\Users\Asus\Documents\3rd year\term 1\FDE-Manan-Gupta\nyc-tlc-cbd-pipeline\class_notebooks\flasheats\database\flasheats.db
Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,None,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order |  |  |  |
| Delivered orders have completion time |  |  |  |
| Promised ETA is valid |  |  |  |
| Event chronology is valid |  |  |  |
| “Late” has an agreed definition |  |  |  |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

### My validation contract

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is unique | rows vs unique `order_id` | **High**: duplicates are double-counted |
| Delivered orders have completion time | delivered ⇒ `actual_delivery_at` not null | null check on delivered rows | **High**: silently shrinks the denominator |
| Promised ETA is valid | `promised_eta` > `created_at` | chronology check | Medium: creates fake extreme delays |
| Event chronology is valid | `created_at` ≤ `pickup_at` ≤ `actual_delivery_at` | chronology check | Medium: impossible durations |
| "Late" has an agreed definition | one owner-approved rule and threshold | read `client_metric_definitions.json` | **High**: changes the headline number |

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

# TODO:
# - establish business grain
# - check delivered orders with missing completion time
# - test promised_eta >= created_at
# - test pickup_at <= actual_delivery_at
# - mark which assumptions need stakeholder clarification

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


In [4]:
o = orders.copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    o[c] = pd.to_datetime(o[c], format="mixed", errors="coerce")
o["status"] = o["final_status"].str.strip().str.lower()

dup_ids = o.loc[o.order_id.duplicated(), "order_id"].tolist()
dup_rows = o[o.order_id.isin(dup_ids)].sort_values("order_id")
conflicting = [c for c in o.columns if dup_rows.groupby("order_id")[c].nunique(dropna=False).max() > 1]

contract = pd.DataFrame([
    ["One row = one business order", f"{len(o) - o.order_id.nunique()} duplicate rows ({', '.join(dup_ids)}); copies differ in {conflicting}"],
    ["Delivered orders have completion time", f"{((o.status == 'delivered') & o.actual_delivery_at.isna()).sum()} delivered orders have no actual_delivery_at"],
    ["Promised ETA is valid", f"{(o.promised_eta < o.created_at).sum()} orders promised before they were created"],
    ["Event chronology is valid", f"{(o.pickup_at < o.created_at).sum()} pickups before creation, {(o.actual_delivery_at < o.pickup_at).sum()} deliveries before pickup"],
    ["'Late' has an agreed definition", metric_notes["note"]],
], columns=["assumption", "what the data shows"])
contract

,assumption,what the data shows
0,One row = one business order,"3 duplicate rows (O00120, O00723, O01302); copies differ in ['traffic_bucket']"
1,Delivered orders have completion time,37 delivered orders have no actual_delivery_at
2,Promised ETA is valid,4 orders promised before they were created
3,Event chronology is valid,"0 pickups before creation, 5 deliveries before pickup"
4,'Late' has an agreed definition,No canonical KPI owner is formally documented.


**What could make 56% misleading:** every assumption in the contract fails at least a little. Duplicates and
capitalised statuses are easy technical fixes. The 37 missing completion times and the chronology errors need a
source decision (Class 5 showed the driver app can fill them). The definition problem is organisational: nobody owns
the KPI, so no amount of cleaning makes "56%" official.

# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [5]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

# TODO: calculate the late rate using different definitions.

In [6]:
analysis["status"] = analysis["final_status"].str.strip().str.lower()
delivered_known = analysis[(analysis.status == "delivered") & analysis.actual_delivery_at.notna()]
not_cancelled = analysis[analysis.status != "cancelled"]
missing = ((analysis.status == "delivered") & analysis.actual_delivery_at.isna()).sum()
late_any = (delivered_known.delay_min > 0).sum()

raw = orders.copy()                               # what an unchecked query on the raw table would give
for c in ["promised_eta", "actual_delivery_at"]:
    raw[c] = pd.to_datetime(raw[c], format="mixed", errors="coerce")
raw_sel = raw[(raw.final_status == "delivered") & raw.actual_delivery_at.notna()]
raw_late = ((raw_sel.actual_delivery_at - raw_sel.promised_eta).dt.total_seconds() > 0)

definitions = pd.DataFrame([
    ["VP Operations: any delay > 0 min", late_any, len(delivered_known),
     "Every missed promise counts, even by one minute"],
    ["Support Lead: delay > 10 min", (delivered_known.delay_min > 10).sum(), len(delivered_known),
     "Delays customers actually notice and complain about"],
    ["Finance: all non-cancelled orders", late_any, len(not_cancelled),
     "Missing delivery times counted as not late (lower bound)"],
    ["Finance, missing times counted as late", late_any + missing, len(not_cancelled),
     "Upper bound if every missing time was a late order"],
    ["Data Team historical: delivered, non-null", late_any, len(delivered_known),
     "Same rule as VP Ops; this is where 56% comes from"],
    ["Raw table as-is (no dedup, case-sensitive)", raw_late.sum(), len(raw_sel),
     "What you get without any validation"],
], columns=["definition", "late", "denominator", "business meaning"])
definitions["late_rate_%"] = (100 * definitions.late / definitions.denominator).round(1)
definitions[["definition", "late_rate_%", "late", "denominator", "business meaning"]]

,definition,late_rate_%,late,denominator,business meaning
0,VP Operations: any delay > 0 min,56.4,843,1495,"Every missed promise counts, even by one minute"
1,Support Lead: delay > 10 min,23.3,349,1495,Delays customers actually notice and complain about
2,Finance: all non-cancelled orders,55.0,843,1532,Missing delivery times counted as not late (lower bound)
3,"Finance, missing times counted as late",57.4,880,1532,Upper bound if every missing time was a late order
4,"Data Team historical: delivered, non-null",56.4,843,1495,Same rule as VP Ops; this is where 56% comes from
5,"Raw table as-is (no dedup, case-sensitive)",56.3,841,1493,What you get without any validation


### Challenge 2 answer

| Definition | Late rate | Business meaning |
|---:|---:|---|
| Any delay > 0 min (VP Ops, and the Data Team's historical rule) | **56.4%** | Every missed promise |
| Delay > 10 min (Support) | **23.3%** | Delays customers notice |
| All non-cancelled orders (Finance) | 55.0% to 57.4% | Depends on what the 37 missing delivery times really were |

The 56% claim comes from the "any delay" rule. It is reproducible, but it is only one of several reasonable answers,
and the Support definition gives less than half of it.

**What should leadership publish?** Not a single unlabelled number. I would publish the >0-minute rate *and* the
>10-minute rate, each with its definition written next to it. **Who owns it:** the VP of Operations should own the
KPI definition, because operations is accountable for delivery performance. Support and Finance should sign off on the
threshold and the cancellation rule, and the Data Team should implement it. Right now the metric notes say *no canonical
KPI owner is documented*, and that has to be fixed first.

# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [7]:
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))

print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))

print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))


 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
handoff         1
Ready           1
READY           1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


In [8]:
def observed(values):
    s = pd.Series(values).dropna()
    clean = s.str.strip().str.lower()
    return pd.DataFrame({"raw": s, "trim+lower": clean}).value_counts().reset_index(name="rows")

for name, values in [("final_status", orders.final_status), ("traffic_bucket", orders.traffic_bucket),
                     ("restaurant status", restaurant_status.status), ("ticket category", tickets.category)]:
    print(name); display(observed(values))

final_status


,raw,trim+lower,rows
0,delivered,delivered,1530
1,cancelled,cancelled,68
2,Delivered,delivered,5


traffic_bucket


,raw,trim+lower,rows
0,medium,medium,635
1,high,high,448
2,low,low,381
3,severe,severe,136
4,HIGH,high,3


restaurant status


,raw,trim+lower,rows
0,preparing,preparing,171
1,handed_off,handed_off,168
2,ready,ready,157
3,ready,ready,2
4,handoff,handoff,1
5,Ready,ready,1
6,READY,ready,1
7,unknown,unknown,1


ticket category


,raw,trim+lower,rows
0,late_delivery,late_delivery,38
1,restaurant_delay,restaurant_delay,37
2,eta_changed,eta_changed,37
3,ready_but_waiting,ready_but_waiting,33
4,status_mismatch,status_mismatch,29
5,driver_not_moving,driver_not_moving,25
6,ETA issue,eta issue,1
7,Late Delivery,late delivery,1
8,late_delivery,late_delivery,1


### Challenge 3 answer

| Field | Representation differences (safe to normalise) | Semantic questions (need an owner) |
|---|---|---|
| `final_status` | `Delivered` → `delivered` (5 rows) | none |
| `traffic_bucket` | `HIGH` → `high` (3 rows) | Who assigns the bucket, and when? The duplicated orders have *different* buckets for the same trip |
| restaurant `status` | `Ready`, `READY`, `ready ` (trailing space) → `ready` | `handoff` vs `handed_off`: probably the same, but restaurant ops must confirm. `unknown` (1 row): what does it mean? |
| ticket `category` | `Late Delivery`, `late_delivery ` → `late_delivery` | `ETA issue`: is it `eta_changed` or something else? Support should map it |

My rule: I only normalise case and whitespace in code. Anything that changes *meaning* goes into a mapping table
that the owning team signs off, instead of being guessed in a notebook.

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [9]:
# Example:
# orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean()

# TODO: calculate mapping coverage for all relationships.

In [10]:
base = orders.drop_duplicates("order_id")
t_unique = tickets.drop_duplicates("ticket_id")

def coverage(child, parent, name, risk):
    total, nulls = len(child), child.isna().sum()
    matched = child.dropna().isin(parent).sum()
    status = "PASS" if matched == total else ("WARN" if matched / total >= 0.98 else "FAIL")
    return [name, total, nulls, round(100 * matched / total, 2), status, risk]

mapping = pd.DataFrame([
    coverage(base.restaurant_id, restaurants.restaurant_id, "orders.restaurant_id → restaurants",
             "3 delivered orders can't be attributed to any restaurant"),
    coverage(base.driver_id, drivers.driver_id, "orders.driver_id → drivers",
             "3 delivered orders have no driver at all"),
    coverage(t_unique.order_id, base.order_id, "tickets.order_id → orders",
             "3 tickets can't be linked to an order"),
    coverage(restaurant_status.order_id, base.order_id, "restaurant_status.order_id → orders",
             "Every status maps, but only 500 of 1,600 orders have a status"),
], columns=["relationship", "rows", "nulls", "coverage_%", "status", "risk"])
display(mapping)

print("Rows with a missing key:")
display(base[base.restaurant_id.isna() | base.driver_id.isna()][["order_id", "restaurant_id", "driver_id", "final_status"]])
rs = restaurant_status.merge(base[["order_id", "restaurant_id"]], on="order_id", suffixes=("", "_orders"))
print("restaurant_status rows whose restaurant disagrees with the order:", (rs.restaurant_id != rs.restaurant_id_orders).sum())
print("Orders that have any restaurant status:", restaurant_status.order_id.nunique(), "of", len(base))

,relationship,rows,nulls,coverage_%,status,risk
0,orders.restaurant_id → restaurants,1600,3,99.81,WARN,3 delivered orders can't be attributed to any restaurant
1,orders.driver_id → drivers,1600,3,99.81,WARN,3 delivered orders have no driver at all
2,tickets.order_id → orders,201,3,98.51,WARN,3 tickets can't be linked to an order
3,restaurant_status.order_id → orders,502,0,100.00,PASS,"Every status maps, but only 500 of 1,600 orders have a status"


Rows with a missing key:


,order_id,restaurant_id,driver_id,final_status
400,O00401,None,D046,delivered
401,O00402,None,D107,delivered
402,O00403,None,D099,delivered
403,O00404,R010,None,delivered
404,O00405,R035,None,delivered
405,O00406,R048,None,delivered


restaurant_status rows whose restaurant disagrees with the order: 0
Orders that have any restaurant status: 500 of 1600


### Challenge 4 answer

Every key that is filled in maps correctly. The gaps are **null keys** (3 restaurants, 3 drivers, 3 tickets) and
**thin coverage**: only 31% of orders have any restaurant status at all.

**Is 1% unmapped acceptable?** For the headline late rate, yes: 3 orders out of 1,495 don't move the number. For a
**restaurant or driver league table** used to hold partners accountable, no. Those orders would disappear from someone's
score, so they need to be fixed or shown as "unattributed". The status table is the bigger issue: 69% of orders have no
restaurant status, so it can't support restaurant-level conclusions at all.

# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [11]:
# TODO:
# Merge restaurant_status with orders on order_id
# Parse timestamps
# Compare last_updated_at with order lifecycle timestamps

In [12]:
st = restaurant_status.copy()
st["status_clean"] = st.status.str.strip().str.lower()
st["updated_at"] = pd.to_datetime(st.last_updated_at, format="mixed", errors="coerce")
life = base[["order_id", "created_at", "pickup_at", "actual_delivery_at"]].copy()
for c in ["created_at", "pickup_at", "actual_delivery_at"]:
    life[c] = pd.to_datetime(life[c], format="mixed", errors="coerce")
f = st.merge(life, on="order_id", how="left")

f["after_pickup"] = f.updated_at > f.pickup_at
f["min_before_pickup"] = (f.pickup_at - f.updated_at).dt.total_seconds() / 60

print("Status updates per order:", st.groupby("order_id").size().value_counts().to_dict(), "(one snapshot, no history)")
print("Exact duplicate status rows:", st.duplicated(["order_id", "status", "last_updated_at"]).sum())
print("Updates timestamped before the order existed:", (f.updated_at < f.created_at).sum())
display(f[f.updated_at < f.created_at][["order_id", "status_clean", "updated_at", "created_at"]]
        .assign(hours_early=lambda x: ((x.created_at - x.updated_at).dt.total_seconds() / 3600).round(1)))

print("Status still 'preparing'/'ready' although the driver had already picked up:",
      (f.status_clean.isin(["preparing", "ready"]) & f.after_pickup).sum())
print("Age of the last status at pickup time (minutes), for updates made before pickup:")
display(f.loc[~f.after_pickup & (f.updated_at >= f.created_at), "min_before_pickup"].describe().round(1))

m = f.merge(restaurants[["restaurant_id", "manual_status_updates"]], on="restaurant_id")
display(m.groupby("manual_status_updates").agg(status_rows=("order_id", "count"),
                                               share_updated_after_pickup=("after_pickup", "mean")).round(2))

Status updates per order: {1: 498, 2: 2} (one snapshot, no history)
Exact duplicate status rows: 2
Updates timestamped before the order existed: 6


,order_id,status_clean,updated_at,created_at,hours_early
5,O01016,handed_off,2026-08-17 08:01:00,2026-08-17 13:46:00,5.8
6,O01395,ready,2026-08-21 09:45:00,2026-08-21 15:27:00,5.7
7,O00925,preparing,2026-08-10 11:20:00,2026-08-10 17:10:00,5.8
8,O00788,ready,2026-08-03 06:10:00,2026-08-03 11:58:00,5.8
9,O01168,ready,2026-08-16 07:49:00,2026-08-16 13:43:00,5.9
501,O00925,preparing,2026-08-10 11:20:00,2026-08-10 17:10:00,5.8


Status still 'preparing'/'ready' although the driver had already picked up: 109
Age of the last status at pickup time (minutes), for updates made before pickup:


count    330.0
mean      14.2
std       10.3
min        0.1
25%        6.3
50%       12.4
75%       19.9
max       71.2
Name: min_before_pickup, dtype: float64

,status_rows,share_updated_after_pickup
manual_status_updates,,
0,311,0.37
1,191,0.27


### Challenge 5 answer

The status feed is **one snapshot per order** for only 500 orders. There's no history, so we never see the
preparing → ready → handed-off sequence. On top of that:
- 109 snapshots still say *preparing* or *ready* even though the driver had already picked the food up (stale).
- 6 rows (5 orders, one row is duplicated) are stamped 5.7–5.9 hours *before* the order existed. That looks like a UTC vs IST timezone bug.
- At pickup, the last status was typically ~12 minutes old, and up to 71 minutes old.

| Use case | Fresh enough? | Why |
|---|---|---|
| Weekly analytics | **WARN** | Fine for rough trends once the timezone rows are fixed, but it covers only 31% of orders |
| Live customer ETA | **FAIL** | A status that's ~12 minutes old on average, and often wrong after pickup, would make the ETA *less* reliable |
| Restaurant accountability | **UNKNOWN** | Needs an agreed SLA (e.g. "mark ready within 2 min") and full status history before anyone is blamed |

Freshness only means something against an SLA, and nobody has defined one yet. Restaurant operations would need to own it.

# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

### My validation gate

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | **WARN** | 3 duplicated orders with conflicting traffic values | De-duplicate on `order_id`, and fix the source that writes the copies |
| Timestamp chronology | **WARN** | 4 ETAs before creation, 5 deliveries before pickup, 37 missing delivery times | Recover the times from driver events; exclude the broken rows with a flag |
| KPI definition | **FAIL** | Four stakeholders, four definitions (23% to 57%), no documented owner | VP Operations to own it; agree the threshold and cancellation rule |
| Category semantics | **WARN** | Case and space variants are safe to fix; `handoff` vs `handed_off`, `unknown` and `ETA issue` are not | Owner-approved mapping table |
| Cross-source mapping | **WARN** | All filled keys map; 3+3+3 null keys; status covers 31% of orders | Show "unattributed" in partner reports |
| Freshness | **FAIL** (live) / **WARN** (weekly) | One stale snapshot per order; 5 orders with timezone-shifted times | Define an SLA; capture full status history |

In [13]:
validation_report = {
    "business_grain": "WARN",
    "timestamp_chronology": "WARN",
    "kpi_definition": "FAIL",
    "category_semantics": "WARN",
    "cross_source_mapping": "WARN",
    "freshness": "FAIL for live ETA, WARN for weekly analytics",
    "publish_56_percent": "NO - not until the KPI owner signs off the definition",
}
pd.Series(validation_report, name="status").to_frame()

,status
business_grain,WARN
timestamp_chronology,WARN
kpi_definition,FAIL
category_semantics,WARN
cross_source_mapping,WARN
freshness,"FAIL for live ETA, WARN for weekly analytics"
publish_56_percent,NO - not until the KPI owner signs off the definition


**Should leadership publish "Late Delivery Rate = 56%" today?** **No.** The number itself can be reproduced
(56.4%), but it isn't publishable because nobody owns the definition, and a reasonable alternative gives 23%.

Before it can be published:
1. The VP of Operations formally owns the KPI and signs off the definition (threshold, cancelled orders, missing times).
2. Duplicates are removed at the source, and the 37 missing delivery times are recovered from the driver app.
3. The chronology errors are flagged and excluded, with the count reported next to the metric.
4. The published number always carries its definition, e.g. "56.4% of delivered orders arrived after the promised
   ETA (23.3% by more than 10 minutes)".

# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**